# 📄 底圖防呆安檢與精準量產雷達 (Phase 1: Pose Auditing)
本 Notebook 專為 Google Colab 環境設計，提供 Google Drive 掛載功能、參數設定表單、GPU 自動偵測，以執行底圖雷達前置安檢。產出結果為後續階段二精準量產使用的純文字「生產工單」，並寫回專案的 Global Cache。

In [ ]:
# @title 1. 掛載 Google Drive 與安裝必要套件
from google.colab import drive
import os

print('🔄 正在掛載 Google Drive...')
drive.mount('/content/drive')

print('✅ 掛載完成！')

In [ ]:
# @title 2. 專案路徑與安檢目標設定 { display-mode: "form" }
import os

# 將專案建立在雲端硬碟的某個資料夾下
PROJECT_ROOT = '/content/drive/MyDrive/app/AI/Lora/Lora_Auto_Curation_Engine' # @param {type:"string"}
FACE_ID = 'Tzuyu' # @param {type:"string"}
# 請填寫「準備接受安檢的新圖片」存放的資料夾名稱 (相對於專案根目錄)，支援多個目錄，請用半形逗號分隔
INPUT_DIR_NAMES = 'target_images/Tzuyu' # @param {type:"string"}
# 請填寫「想要排除安檢的子資料夾名稱」，支援多個名稱，請用半形逗號分隔
EXCLUDE_DIR_NAMES = '' # @param {type:"string"}
# 請填寫「每個工單最多包含的圖片數量 (切割門檻)」
TASK_CHUNK_SIZE = 100 # @param {type:"integer"}

input_dir_list = [os.path.join(PROJECT_ROOT, d.strip()) for d in INPUT_DIR_NAMES.split(',') if d.strip()]
exclude_dir_list = [d.strip() for d in EXCLUDE_DIR_NAMES.split(',') if d.strip()]

# === 系統相依目錄 (不可隨意更動) ===
GOLDEN_DIR = os.path.join(PROJECT_ROOT, 'LoRA_Curated_Dataset', FACE_ID, 'Accepted')
REGISTRY_PATH = os.path.join(PROJECT_ROOT, 'lora_registry.json')
MAPPING_JSON_PATH = os.path.join(PROJECT_ROOT, 'mapping.json')
TASK_OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')
GLOBAL_POSE_CACHE_FILE = os.path.join(TASK_OUTPUT_DIR, 'registry', 'global_pose_audit.json')
TASK_LIST_DIR = os.path.join(TASK_OUTPUT_DIR, 'task_lists', FACE_ID)

MIN_FACE_RATIO = 0.033
DET_THRESH = 0.6
MODEL_NAME = 'buffalo_l'

print(f"📁 專案根目錄: {PROJECT_ROOT}")
print(f"🎯 目標 Face ID: {FACE_ID}")
print(f"📥 待檢定圖片目錄列表 (Inputs): {input_dir_list}")
print(f"🚫 排除掃描之子目錄列表 (Excludes): {exclude_dir_list}")
print(f"🌟 黃金標準目錄 (Golden): {GOLDEN_DIR}")
print(f"📥 專案輸出總目錄: {TASK_OUTPUT_DIR}")
print(f"📥 全域依賴FACE_ID底圖位置記錄檔: {GLOBAL_POSE_CACHE_FILE}")
print(f"📥 工單目錄: {TASK_LIST_DIR}")

if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(f"❌ 找不到專案根目錄: {PROJECT_ROOT} (請確認 Drive 是否掛載或路徑是否正確)")

valid_input_dirs = []
for d in input_dir_list:
    if not os.path.exists(d):
        print(f"⚠️ 警告: 找不到待檢定圖片目錄 {d}")
    else:
        valid_input_dirs.append(d)
if not valid_input_dirs:
    raise FileNotFoundError("❌ 所有指定的待檢定圖片目錄皆不存在，請重新確認！")

if not os.path.exists(GOLDEN_DIR):
    raise FileNotFoundError(f"❌ 找不到黃金標準目錄: {GOLDEN_DIR} (請先完成第一階段 Auto Curation 以建立標準)")

os.makedirs(os.path.dirname(GLOBAL_POSE_CACHE_FILE), exist_ok=True)
os.makedirs(TASK_LIST_DIR, exist_ok=True)
print('\n✅ 目錄結構確認完成。請確保目標圖片已放置於待檢定目錄中！')

In [ ]:
# @title 3. 核心安檢雷達執行與產出工單
import sys
import json
import datetime
import re

def load_global_cache():
    if not os.path.exists(GLOBAL_POSE_CACHE_FILE):
        return {}
    try:
        with open(GLOBAL_POSE_CACHE_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return {}

def save_global_cache(cache_data):
    with open(GLOBAL_POSE_CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(cache_data, f, indent=4, ensure_ascii=False)

def get_physical_limits():
    if not os.path.exists(REGISTRY_PATH): return None
    try:
        with open(REGISTRY_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if 'faces' in data:
                for face in data['faces']:
                    if face.get('face_id') == FACE_ID and 'physical_limits' in face:
                        return face['physical_limits']
    except: pass
    return None

def save_physical_limits(limits):
    try:
        if not os.path.exists(REGISTRY_PATH):
            data = {'faces': []}
        else:
            with open(REGISTRY_PATH, 'r', encoding='utf-8') as f:
                data = json.load(f)
        
        if 'faces' not in data:
            data['faces'] = []
            
        face_found = False
        for face in data['faces']:
            if face.get('face_id') == FACE_ID:
                face['physical_limits'] = limits
                face_found = True
                break
                
        if not face_found:
            data['faces'].append({'face_id': FACE_ID, 'name': FACE_ID, 'physical_limits': limits})
            
        json_str = json.dumps(data, indent=4, ensure_ascii=False)
        with open(REGISTRY_PATH, 'w', encoding='utf-8') as f:
            f.write(json_str)
        print(f'💾 動態計算之極限邊界已寫回註冊表。')
    except Exception as e:
        print(f'⚠️ 寫回註冊表失敗: {e}')

def build_dynamic_limits_if_needed(app):
    if not os.path.exists(GOLDEN_DIR):
        print(f'❌ [錯誤] 找不到黃金訓練圖目錄來建立基準: {GOLDEN_DIR}')
        sys.exit(1)
        
    print(f'🔍 正在掃描 {FACE_ID} 的黃金訓練圖以建立絕對物理邊界...')
    aspect_ratios, yaws, pitches, rolls = [], [], [], []
    
    for filename in os.listdir(GOLDEN_DIR):
        if not filename.lower().endswith(('.jpg', '.png', '.jpeg', '.webp')): continue
        img = cv2.imread(os.path.join(GOLDEN_DIR, filename))
        if img is None: continue
        
        faces = app.get(img)
        if len(faces) == 0: continue
        
        face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)[0]
        w, h = face.bbox[2] - face.bbox[0], face.bbox[3] - face.bbox[1]
        aspect_ratios.append(w / h)
        pitch, yaw, roll = face.pose
        pitches.append(pitch); yaws.append(yaw); rolls.append(roll)
        
    if not aspect_ratios:
        print('❌ [錯誤] 無法從黃金圖中提取特徵。')
        sys.exit(1)
        
    limits = {
        'aspect_ratio': [round(float(min(aspect_ratios) * 0.9), 2), round(float(max(aspect_ratios) * 1.1), 2)],
        'yaw': [round(float(min(yaws) - 10), 2), round(float(max(yaws) + 10), 2)],
        'pitch': [round(float(min(pitches) - 10), 2), round(float(max(pitches) + 10), 2)],
        'roll': [round(float(min(rolls) - 10), 2), round(float(max(rolls) + 10), 2)]
    }
    print(f'✅ 動態邊界建立完成: {limits}')
    return limits

# 1. 輕量盤點 (純 CPU 字串比對)
print(f'\n📂 開始盤點目標目錄列表...')
cache = load_global_cache()
if FACE_ID not in cache:
    cache[FACE_ID] = {'accept': [], 'reject': []}
    
face_cache = cache[FACE_ID]
accepted_paths = set(face_cache['accept'])
rejected_paths = set(face_cache['reject'])

all_files = []
for in_dir in valid_input_dirs:
    for root, dirs, files in os.walk(in_dir):
        for ex_dir in exclude_dir_list:
            if ex_dir in dirs:
                dirs.remove(ex_dir)
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg', '.webp')):
                all_files.append(os.path.join(root, f))

mapping_data = {}
if os.path.exists(MAPPING_JSON_PATH):
    try:
        with open(MAPPING_JSON_PATH, 'r', encoding='utf-8') as f:
            mapping_data = json.load(f)
    except Exception as e:
        print(f'⚠️ [警告] 讀取 mapping.json 失敗: {e}')
else:
    print(f'⚠️ [警告] 找不到 mapping.json: {MAPPING_JSON_PATH}')

invalid_files = []
file_face_indices = {}
valid_all_files = []
for fpath in all_files:
    fname = os.path.basename(fpath)
    if fname in mapping_data:
        file_face_indices[fname] = mapping_data[fname]
        valid_all_files.append(fpath)
    else:
        invalid_files.append(fpath)

all_files = valid_all_files

if invalid_files:
    print(f'\n⚠️ 發現 {len(invalid_files)} 張圖片在 mapping.json 中找不到對應的臉部索引！')
    choice = input('請問要中斷執行並匯出錯誤 Log (輸入 \'Q\')，還是忽略這些錯誤檔案繼續 (輸入 \'C\')？ ').strip().upper()
    if choice == 'Q':
        log_path = os.path.join(PROJECT_ROOT, 'missing_mapping_log.txt')
        with open(log_path, 'w', encoding='utf-8') as lf:
            for inv_f in invalid_files: lf.write(inv_f + '\n')
        print(f'🛑 已中斷執行。錯誤清單已匯出至 {log_path}')
        sys.exit(1)
    else:
        print('▶️ 將忽略這些錯誤檔案，繼續執行。')

history_accept, history_reject, unknown = [], [], []
for fpath in all_files:
    abs_path = os.path.abspath(fpath)
    if abs_path in accepted_paths: history_accept.append(abs_path)
    elif abs_path in rejected_paths: history_reject.append(abs_path)
    else: unknown.append(abs_path)

print(f'\n📊 盤點戰情報告:\n✅ 歷史已接受: {len(history_accept)} 張\n❌ 歷史已拒絕: {len(history_reject)} 張\n❓ 未知待檢定: {len(unknown)} 張\n')

if len(history_accept) > 0 or len(history_reject) > 0:
    print(f'\n👉 系統偵測到有 {len(history_accept) + len(history_reject)} 張圖片已在過去安檢完成 (剩餘 {len(unknown)} 張未處理)。')
    reset_choice = input('請問您要「保留進度」(輸入 n)，還是要「全數覆蓋，重新掃描全部」(輸入 y)？ [預設保留: n]: ')
    if reset_choice.strip().lower() == 'y':
        print('🔄 已清除歷史紀錄，所有圖片將重新進入 GPU 檢定流程。')
        unknown.extend(history_accept + history_reject)
        history_accept, history_reject = [], []
        cache[FACE_ID] = {'accept': [], 'reject': []}
        save_global_cache(cache)

if len(unknown) > 0:
    choice = input(f'\n👉 是否準備啟動 GPU 載入模型，檢定這 {len(unknown)} 張未知圖片？(Y/n): ')
    if choice.strip().lower() == 'n':
        print('🚫 已取消 GPU 掃描。')
        unknown = []

if len(unknown) > 0:
    print('📦 正在動態安裝 GPU 與視覺相關套件 (這可能需要幾十秒鐘)...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'insightface', 'onnxruntime-gpu', 'opencv-python', 'numpy'])
    import cv2
    import numpy as np
    import onnxruntime as ort
    import insightface
    from insightface.app import FaceAnalysis
    from tqdm import tqdm
    print('🤖 載入 InsightFace 模型進行未知圖片精準掃描...')
    providers = [('CUDAExecutionProvider', {'arena_extend_strategy': 'kSameAsRequested'}), 'CPUExecutionProvider']
    app = FaceAnalysis(name=MODEL_NAME, providers=providers)
    app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=DET_THRESH)
    
    limits = get_physical_limits()
    if not limits:
        print('⚠️ 註冊表中無 physical_limits，嘗試動態掃描黃金訓練圖...')
        limits = build_dynamic_limits_if_needed(app)
        save_physical_limits(limits)
        
    print(f'\n🔍 開始物理極限精準掃描 {len(unknown)} 張未知圖片...')
    print('>> 為避免畫面洗版，詳細明細將寫入 Log 檔，此處僅顯示進度。')
    new_accept, new_reject = [], []
    batch_accept, batch_reject = [], []
    SAVE_INTERVAL = 100
    
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_filename = f'audit_log_{FACE_ID}_{timestamp}.txt'
    log_filepath = os.path.join(TASK_LIST_DIR, log_filename)
    
    with open(log_filepath, 'w', encoding='utf-8') as log_f:
        log_f.write(f'=== {FACE_ID} 底圖安檢詳細紀錄 ===\n\n')
        log_f.write(f'【本次安檢合格基準 (Dynamic Limits)】\n')
        log_f.write(f' - 最低臉部佔比要求: {MIN_FACE_RATIO*100:.1f}%\n')
        log_f.write(f' - 臉型比例 (Aspect Ratio): {limits["aspect_ratio"][0]:.2f} ~ {limits["aspect_ratio"][1]:.2f}\n')
        log_f.write(f' - 左右轉頭 (Yaw) 容許範圍: {limits["yaw"][0]:.1f}度 ~ {limits["yaw"][1]:.1f}度\n')
        log_f.write(f' - 上下仰俯 (Pitch) 容許範圍: {limits["pitch"][0]:.1f}度 ~ {limits["pitch"][1]:.1f}度\n')
        log_f.write(f' - 歪頭傾斜 (Roll) 容許範圍: {limits["roll"][0]:.1f}度 ~ {limits["roll"][1]:.1f}度\n\n')
        log_f.write(f'【圖片明細】\n')
        
        for idx, fpath in enumerate(tqdm(unknown, desc='⏳ 掃描進度')):
            img = cv2.imread(fpath)
            if img is None:
                new_reject.append(fpath)
                log_f.write(f'❌ [Reject] {fpath} -> 原因: 無法讀取圖片\n')
                continue
                
            faces = app.get(img)
            if len(faces) == 0:
                 new_reject.append(fpath)
                 log_f.write(f'❌ [Reject] {fpath} -> 原因: 找不到人臉\n')
                 continue
                 
            fname = os.path.basename(fpath)
            target_face_idx = file_face_indices.get(fname, 0)
            if target_face_idx < 0 or target_face_idx >= len(faces):
                 new_reject.append(fpath)
                 log_f.write(f'❌ [Reject] {fpath} -> 原因: 找不到對應的臉部索引 {target_face_idx} (僅偵測到 {len(faces)} 張臉)\n')
                 continue
                 
            face = faces[target_face_idx]
            h, w_img = img.shape[:2]
            face_ratio = ((face.bbox[2]-face.bbox[0])*(face.bbox[3]-face.bbox[1])) / (h*w_img)
            ar = (face.bbox[2]-face.bbox[0]) / (face.bbox[3]-face.bbox[1])
            pitch, yaw, roll = face.pose
            
            reasons = []
            if face_ratio < MIN_FACE_RATIO:
                reasons.append(f'佔比太小 {face_ratio*100:.1f}%')
            if not (limits['aspect_ratio'][0] <= ar <= limits['aspect_ratio'][1]):
                reasons.append(f'臉型比例不符 {ar:.2f}')
            if not (limits['yaw'][0] <= yaw <= limits['yaw'][1]):
                reasons.append(f'左右轉頭超標 Yaw:{yaw:.1f}')
            if not (limits['pitch'][0] <= pitch <= limits['pitch'][1]):
                reasons.append(f'上下仰俯超標 Pitch:{pitch:.1f}')
            if not (limits['roll'][0] <= roll <= limits['roll'][1]):
                reasons.append(f'歪頭超標 Roll:{roll:.1f}')
                
            if reasons:
                batch_reject.append(fpath)
                log_f.write(f'❌ [Reject] {fpath}\n')
                log_f.write(f'   -> 綜合失敗原因: {", ".join(reasons)}\n')
            else:
                batch_accept.append(fpath)
                log_f.write(f'✅ [Accept] {fpath}\n')
                log_f.write(f'   -> 數據: 佔比:{face_ratio*100:.1f}%, 比例:{ar:.2f}, Yaw:{yaw:.1f}, Pitch:{pitch:.1f}, Roll:{roll:.1f}\n')
                
            if (idx + 1) % SAVE_INTERVAL == 0:
                cache[FACE_ID]['accept'].extend(batch_accept)
                cache[FACE_ID]['reject'].extend(batch_reject)
                save_global_cache(cache)
                new_accept.extend(batch_accept)
                new_reject.extend(batch_reject)
                batch_accept, batch_reject = [], []
                
    if batch_accept or batch_reject:
        cache[FACE_ID]['accept'].extend(batch_accept)
        cache[FACE_ID]['reject'].extend(batch_reject)
        save_global_cache(cache)
        new_accept.extend(batch_accept)
        new_reject.extend(batch_reject)

    print(f'\n✅ 掃描完成！單筆詳細紀錄已匯出至: {log_filepath}')
    print('\n💾 所有盤點紀錄已完整寫回 Global Cache。')
else:
    new_accept = []
    print('🎉 無未知圖片，跳過 GPU 模型檢定，將直接使用歷史合格紀錄打包新工單。')

all_task_accept = history_accept + new_accept
if all_task_accept:
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    print('\n' + '='*50)
    print(f'🧾 工單產出完成! 共計 {len(all_task_accept)} 張合格底圖。')
    
    if len(all_task_accept) > TASK_CHUNK_SIZE:
        print(f'📦 由於數量超過門檻 ({TASK_CHUNK_SIZE})，將自動切割成多份工單檔案...')
        for i in range(0, len(all_task_accept), TASK_CHUNK_SIZE):
            chunk = all_task_accept[i:i + TASK_CHUNK_SIZE]
            chunk_idx = (i // TASK_CHUNK_SIZE) + 1
            task_filename = f'task_{FACE_ID}_{timestamp}_{chunk_idx}.txt'
            task_filepath = os.path.join(TASK_LIST_DIR, task_filename)
            with open(task_filepath, 'w', encoding='utf-8') as f:
                for p in chunk: f.write(f'{p}\n')
            print(f'  -> 產出: {task_filename} ({len(chunk)} 張)')
    else:
        task_filename = f'task_{FACE_ID}_{timestamp}.txt'
        task_filepath = os.path.join(TASK_LIST_DIR, task_filename)
        with open(task_filepath, 'w', encoding='utf-8') as f:
            for p in all_task_accept: f.write(f'{p}\n')
        print(f'  -> 產出: {task_filename} ({len(all_task_accept)} 張)')
    print('='*50)
else:
    print('⚠️ 本次任務沒有任何合格底圖可產出工單。')